In [5]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [7]:
import pandas as pd
import re

#F = Mensagens seguras T = Mensagens problemáticas
# 1. Carregar os dados
df_comentarios = pd.read_csv('./datasets/finalcomments30up.csv', names=['id', 'texto', 'autor', 'comunidade'])
df_rotulos = pd.read_csv('./datasets/manualjudgement.csv', names=['id', 'label'])

# 2. Unir as bases
df_consolidado = pd.merge(df_comentarios, df_rotulos, on='id')

# 3. Função de Adaptação para Controle Parental
def detectar_risco_grooming(row):
    texto = str(row['texto']).lower()
    label_original = row['label']
    
    # Procuramos por http, https, www, .com, .br, .net, .org, .gov
    padrao_url = r'(http|https|www|\.com|\.br|\.net|\.org|\.gov)'
    
    if re.search(padrao_url, texto):
        return 'T' # Força o rótulo para Tóxico/Malicioso
    
    return label_original

# Aplicando lógica
df_consolidado['label'] = df_consolidado.apply(detectar_risco_grooming, axis=1)

# 4. Salvar o novo dataset mestre
df_consolidado.to_csv('dataset_parental_final.csv', index=False)

print("União e Adaptação concluídas!")
print(f"Total de mensagens: {len(df_consolidado)}")
print("\nNova distribuição após considerar URLs como risco:")
print(df_consolidado['label'].value_counts())

União e Adaptação concluídas!
Total de mensagens: 19272

Nova distribuição após considerar URLs como risco:
label
F    14848
T     4424
Name: count, dtype: int64
